In [13]:
# --- imports ---
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# ============================================================
sorted_symbols_list = ["RSP", "SPY"]
filename = f"{'_'.join(sorted_symbols_list)}.csv"

base_directory = Path("..").resolve()

input_path = base_directory / "historical prices" / filename
output_path = base_directory / "backtests" / filename


async def main():

    df = pd.read_csv(input_path, index_col=0)
    print(df)

    df.reset_index(names="eop date", inplace=True)
    df.insert(0, "bop date", df["eop date"].shift(1))

    for symbol in sorted_symbols_list:
        bop_price = f"bop {symbol} price"
        eop_price = f"eop {symbol} price"

        df[bop_price] = df[symbol].shift(1)
        df[eop_price] = df[symbol]
        df[f"{symbol} cop"] = df[eop_price] - df[bop_price]
        df[f"{symbol} pct cop"] = np.log(df[eop_price] / df[bop_price])
        df.drop(columns=symbol, inplace=True)

    df.to_csv(output_path)
    print(f"Saved {output_path}")
    print("finished")


await main()


               RSP     SPY
date                      
2006-08-30   43.43  130.74
2006-08-31   43.48  130.74
2006-09-01   43.68  131.44
2006-09-05   43.75  131.58
2006-09-06   43.24  130.46
...            ...     ...
2026-08-17  220.79  772.67
2026-08-18  219.79  767.45
2026-08-19  222.07  769.06
2026-08-20  220.28  762.60
2026-08-21  221.67  765.72

[5024 rows x 2 columns]
Saved C:\Users\micha\git-projects\backtesting\backtests\RSP_SPY.csv
finished
